In [2]:
!pip install --upgrade pip setuptools wheel


In [3]:
!pip install pyarrow --only-binary=:all:


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 MB 78.3 MB/s  0:00:00 eta 0:00:01


In [5]:
!pip install torch transformers accelerate sentencepiece
!pip install datasets pyarrow==14.0.2 --only-binary=:all:


  Using cached datasets-4.4.2-py3-none-any.whl.metadata (19 kB)
INFO: pip is looking at multiple versions of datasets to determine which version is compatible with other requirements. This could take a while.
  Using cached datasets-4.4.1-py3-none-any.whl.metadata (19 kB)
  Using cached datasets-4.4.0-py3-none-any.whl.metadata (19 kB)
  Using cached datasets-4.3.0-py3-none-any.whl.metadata (18 kB)
  Using cached datasets-4.2.0-py3-none-any.whl.metadata (18 kB)
  Using cached datasets-4.1.1-py3-none-any.whl.metadata (18 kB)
  Using cached datasets-4.1.0-py3-none-any.whl.metadata (18 kB)
  Using cached datasets-4.0.0-py3-none-any.whl.metadata (19 kB)
INFO: pip is still looking at multiple versions of datasets to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidanc

In [6]:
import pyarrow
import datasets

print(pyarrow.__version__)
print(datasets.__version__)


<frozen importlib._bootstrap>:241: RuntimeWarning: pyarrow.lib.Tensor size changed, may indicate binary incompatibility. Expected 64 from C header, got 80 from PyObject
<frozen importlib._bootstrap>:241: RuntimeWarning: pyarrow.lib.ChunkedArray size changed, may indicate binary incompatibility. Expected 64 from C header, got 72 from PyObject
<frozen importlib._bootstrap>:241: RuntimeWarning: pyarrow.lib._Tabular size changed, may indicate binary incompatibility. Expected 24 from C header, got 32 from PyObject
<frozen importlib._bootstrap>:241: RuntimeWarning: pyarrow.lib.Table size changed, may indicate binary incompatibility. Expected 56 from C header, got 64 from PyObject
<frozen importlib._bootstrap>:241: RuntimeWarning: pyarrow.lib.NativeFile size changed, may indicate binary incompatibility. Expected 96 from C header, got 104 from PyObject
<frozen importlib._bootstrap>:241: RuntimeWarning: pyarrow.lib.BufferedInputStream size changed, may indicate binary incompatibility. Expected 

20.0.0
2.19.2


In [3]:
# ============================================================
# FINAL HYDRALORA SCRIPT — NAN-SAFE, STABLE, SINGLE BLOCK
# Mistral-7B | A100 80GB | BF16 | Research-Grade
# ============================================================

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:64"

import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from torch.utils.data import DataLoader

device = "cuda"
torch.backends.cuda.matmul.allow_tf32 = True

# =========================
# Tokenizer
# =========================
model_name = "mistralai/Mistral-7B-v0.1"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=False
)
tokenizer.pad_token = tokenizer.eos_token

# =========================
# Model
# =========================
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

model.config.pad_token_id = tokenizer.eos_token_id
model.config.use_flash_attention_2 = True
model.config.use_cache = False

model.gradient_checkpointing_enable()
model.enable_input_require_grads()

# =========================
# HydraLoRA
# =========================
class HydraLoRA(nn.Module):
    def __init__(self, in_features, out_features, r=8, num_experts=8, alpha=32, dropout=0.05):
        super().__init__()
        self.scaling = alpha / r

        self.lora_A = nn.ModuleList([
            nn.Linear(in_features, r, bias=False)
            for _ in range(num_experts)
        ])
        self.lora_B = nn.ModuleList([
            nn.Linear(r, out_features, bias=False)
            for _ in range(num_experts)
        ])

        self.router = nn.Linear(in_features, num_experts)
        self.dropout = nn.Dropout(dropout)

        for A, B in zip(self.lora_A, self.lora_B):
            nn.init.kaiming_uniform_(A.weight, a=5 ** 0.5)
            nn.init.zeros_(B.weight)

        nn.init.zeros_(self.router.weight)
        nn.init.zeros_(self.router.bias)

    def forward(self, x):
        # ---- Stable router ----
        logits = self.router(x)
        logits = logits - logits.max(dim=-1, keepdim=True).values
        logits = torch.clamp(logits, -20, 20)
        gates = F.softmax(logits, dim=-1)

        expert_outs = []
        for A, B in zip(self.lora_A, self.lora_B):
            expert_outs.append(B(self.dropout(A(x))) * self.scaling)

        expert_outs = torch.stack(expert_outs, dim=-2)
        return torch.sum(gates.unsqueeze(-1) * expert_outs, dim=-2)

class HydraLoRALinear(nn.Module):
    def __init__(self, base_layer):
        super().__init__()
        self.base = base_layer

        self.hydra = HydraLoRA(
            base_layer.in_features,
            base_layer.out_features
        )

        # Device + dtype alignment
        self.hydra.to(
            device=base_layer.weight.device,
            dtype=base_layer.weight.dtype
        )

        for p in self.base.parameters():
            p.requires_grad = False

    def forward(self, x):
        return self.base(x) + self.hydra(x)

# =========================
# Inject HydraLoRA into Q,V
# =========================
def apply_hydralora(model):
    for name, module in model.named_modules():
        if name.endswith("q_proj") or name.endswith("v_proj"):
            parent = model
            *path, attr = name.split(".")
            for p in path:
                parent = getattr(parent, p)
            setattr(parent, attr, HydraLoRALinear(getattr(parent, attr)))

apply_hydralora(model)

# =========================
# Freeze non-Hydra params
# =========================
for name, param in model.named_parameters():
    param.requires_grad = ("hydra" in name.lower())

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,}")
print(f"Trainable %: {100 * trainable / total:.4f}%")

# =========================
# Dataset
# =========================
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")

def tokenize_fn(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128,
        padding=False
    )

tokenized = dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=["text"]
)
tokenized.set_format(type="torch")

def collate_fn(batch):
    return {
        "input_ids": torch.nn.utils.rnn.pad_sequence(
            [x["input_ids"] for x in batch],
            batch_first=True,
            padding_value=tokenizer.pad_token_id
        )
    }

train_loader = DataLoader(
    tokenized["train"],
    batch_size=1,
    shuffle=True,
    collate_fn=collate_fn
)

# =========================
# Optimizer (stable LR)
# =========================
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=5e-5
)

# =========================
# Training (NaN-SAFE)
# =========================
model.train()

GRAD_ACC = 8
MAX_STEPS = 500
step = 0
skipped = 0

for batch in train_loader:
    input_ids = batch["input_ids"].to(device)

    # ---- Skip empty / tiny batches ----
    if input_ids.numel() == 0 or input_ids.shape[1] < 2:
        skipped += 1
        continue

    outputs = model(input_ids=input_ids, labels=input_ids)
    loss = outputs.loss

    # ---- Skip NaN / Inf losses ----
    if torch.isnan(loss) or torch.isinf(loss):
        optimizer.zero_grad(set_to_none=True)
        skipped += 1
        continue

    loss = loss / GRAD_ACC
    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        filter(lambda p: p.requires_grad, model.parameters()),
        max_norm=1.0
    )

    if (step + 1) % GRAD_ACC == 0:
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)

    if step % 50 == 0:
        print(f"Step {step} | Loss: {loss.item() * GRAD_ACC:.4f}")

    step += 1
    if step >= MAX_STEPS:
        break

print(f"✅ Training finished | Skipped batches: {skipped}")


Loading checkpoint shards: 100%|██████████| 2/2 [00:04<00:00,  2.40s/it]


Trainable params: 29,360,640
Trainable %: 0.4038%
Step 0 | Loss: 5.5946
Step 50 | Loss: 2.6866
Step 100 | Loss: 2.0988
Step 150 | Loss: 2.2924
Step 200 | Loss: 1.6744
Step 250 | Loss: 2.8313
Step 300 | Loss: 1.6380
Step 350 | Loss: 1.2959
Step 400 | Loss: 1.8854
Step 450 | Loss: 5.6311
✅ Training finished | Skipped batches: 252


In [4]:
# Router entropy (lower over time = specialization)
with torch.no_grad():
    x = torch.randn(2, 16, model.config.hidden_size, device="cuda", dtype=torch.bfloat16)
    logits = model.model.layers[0].self_attn.q_proj.hydra.router(x)
    probs = torch.softmax(logits, dim=-1)
    entropy = -(probs * torch.log(probs + 1e-8)).sum(dim=-1).mean()
    print("Router entropy:", entropy.item())


Router entropy: 2.078125


PAPER-FAITHFUL HYDRALORA CODE

In [1]:
# ============================================================
# PAPER-FAITHFUL HYDRALORA (TOP-2 ROUTING + ENTROPY METRICS)
# Mistral-7B | A100 80GB | BF16 | Research-grade
# ============================================================

import os, math
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:64"

import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from torch.utils.data import DataLoader

# -------------------------
# Global setup
# -------------------------
device = "cuda"
torch.backends.cuda.matmul.allow_tf32 = True

model_name = "mistralai/Mistral-7B-v0.1"

# -------------------------
# Tokenizer
# -------------------------
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=False
)
tokenizer.pad_token = tokenizer.eos_token

# -------------------------
# Model
# -------------------------
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

model.config.pad_token_id = tokenizer.eos_token_id
model.config.use_flash_attention_2 = True
model.config.use_cache = False

model.gradient_checkpointing_enable()
model.enable_input_require_grads()

# ============================================================
# HydraLoRA (Top-2 Sparse Routing)
# ============================================================
class HydraLoRA(nn.Module):
    def __init__(
        self,
        in_features,
        out_features,
        r=8,
        num_experts=8,
        alpha=32,
        dropout=0.05,
        top_k=2
    ):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.scaling = alpha / r

        self.lora_A = nn.ModuleList([
            nn.Linear(in_features, r, bias=False)
            for _ in range(num_experts)
        ])
        self.lora_B = nn.ModuleList([
            nn.Linear(r, out_features, bias=False)
            for _ in range(num_experts)
        ])

        self.router = nn.Linear(in_features, num_experts)
        self.dropout = nn.Dropout(dropout)

        for A, B in zip(self.lora_A, self.lora_B):
            nn.init.kaiming_uniform_(A.weight, a=5 ** 0.5)
            nn.init.zeros_(B.weight)

        nn.init.zeros_(self.router.weight)
        nn.init.zeros_(self.router.bias)

    def forward(self, x):
        # Router logits (stable)
        logits = self.router(x)
        logits = logits - logits.max(dim=-1, keepdim=True).values
        logits = torch.clamp(logits, -20, 20)

        # Top-2 routing
        topk_vals, topk_idx = torch.topk(logits, k=self.top_k, dim=-1)
        gates = F.softmax(topk_vals, dim=-1)

        # Expert outputs
        expert_outs = []
        for A, B in zip(self.lora_A, self.lora_B):
            expert_outs.append(
                B(self.dropout(A(x))) * self.scaling
            )
        expert_outs = torch.stack(expert_outs, dim=-2)

        # Sparse combine
        out = torch.zeros_like(expert_outs[..., 0, :])
        for i in range(self.top_k):
            idx = topk_idx[..., i]
            gate = gates[..., i].unsqueeze(-1)
            selected = torch.gather(
                expert_outs,
                dim=-2,
                index=idx.unsqueeze(-1).unsqueeze(-1)
                    .expand(-1, -1, 1, expert_outs.size(-1))
            ).squeeze(-2)
            out += gate * selected

        return out

class HydraLoRALinear(nn.Module):
    def __init__(self, base_layer):
        super().__init__()
        self.base = base_layer
        self.hydra = HydraLoRA(
            base_layer.in_features,
            base_layer.out_features
        ).to(
            device=base_layer.weight.device,
            dtype=base_layer.weight.dtype
        )

        for p in self.base.parameters():
            p.requires_grad = False

    def forward(self, x):
        return self.base(x) + self.hydra(x)

# -------------------------
# Inject into Q, V projections
# -------------------------
def apply_hydralora(model):
    for name, module in model.named_modules():
        if name.endswith("q_proj") or name.endswith("v_proj"):
            parent = model
            *path, attr = name.split(".")
            for p in path:
                parent = getattr(parent, p)
            setattr(parent, attr, HydraLoRALinear(getattr(parent, attr)))

apply_hydralora(model)

# -------------------------
# Freeze non-Hydra params
# -------------------------
for n, p in model.named_parameters():
    p.requires_grad = ("hydra" in n.lower())

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,}")
print(f"Trainable %: {100 * trainable / total:.4f}%")

# ============================================================
# Dataset
# ============================================================
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")

def tokenize_fn(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128,
        padding=False
    )

tokenized = dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=["text"]
)
tokenized.set_format(type="torch")

def collate_fn(batch):
    return {
        "input_ids": torch.nn.utils.rnn.pad_sequence(
            [x["input_ids"] for x in batch],
            batch_first=True,
            padding_value=tokenizer.pad_token_id
        )
    }

train_loader = DataLoader(
    tokenized["train"],
    batch_size=1,
    shuffle=True,
    collate_fn=collate_fn
)

# -------------------------
# Optimizer
# -------------------------
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=5e-5
)

# ============================================================
# Training loop
# ============================================================
model.train()

GRAD_ACC = 8
MAX_STEPS = 500
step = 0
skipped = 0

for batch in train_loader:
    input_ids = batch["input_ids"].to(device)

    if input_ids.numel() == 0 or input_ids.shape[1] < 2:
        skipped += 1
        continue

    outputs = model(input_ids=input_ids, labels=input_ids)
    loss = outputs.loss

    if torch.isnan(loss) or torch.isinf(loss):
        optimizer.zero_grad(set_to_none=True)
        skipped += 1
        continue

    loss = loss / GRAD_ACC
    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        filter(lambda p: p.requires_grad, model.parameters()),
        max_norm=1.0
    )

    if (step + 1) % GRAD_ACC == 0:
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)

    if step % 50 == 0:
        print(f"Step {step} | Loss: {loss.item() * GRAD_ACC:.4f}")

    step += 1
    if step >= MAX_STEPS:
        break

print(f"✅ Training finished | Skipped batches: {skipped}")

# ============================================================
# ENTROPY METRICS (CORRECT, PAPER-FAITHFUL)
# ============================================================

num_experts = 8
expert_counts = torch.zeros(num_experts, device="cuda")

with torch.no_grad():
    for _ in range(200):
        x = torch.randn(
            4, 32, model.config.hidden_size,
            device="cuda", dtype=torch.bfloat16
        )
        logits = model.model.layers[0].self_attn.q_proj.hydra.router(x)
        _, topk_idx = torch.topk(logits, k=2, dim=-1)

        for i in range(2):
            expert_counts.scatter_add_(
                0,
                topk_idx[..., i].reshape(-1),
                torch.ones_like(
                    topk_idx[..., i].reshape(-1),
                    dtype=torch.float
                )
            )

probs = expert_counts / expert_counts.sum()
expert_entropy = -(probs * torch.log(probs + 1e-8)).sum()

print("\n=== ROUTING DIAGNOSTICS ===")
print("Expert usage probs:", probs.tolist())
print(f"Expert-usage entropy: {expert_entropy.item():.4f}")
print(f"Max possible entropy: {math.log(num_experts):.4f}")

# Optional: top-2 routing entropy
top2_entropy_vals = []
with torch.no_grad():
    x = torch.randn(
        4, 32, model.config.hidden_size,
        device="cuda", dtype=torch.bfloat16
    )
    logits = model.model.layers[0].self_attn.q_proj.hydra.router(x)
    topk_vals, _ = torch.topk(logits, k=2, dim=-1)
    probs = torch.softmax(topk_vals, dim=-1)
    top2_entropy = -(probs * torch.log(probs + 1e-8)).sum(dim=-1).mean()

print(f"Top-2 routing entropy (max log2≈0.693): {top2_entropy.item():.4f}")

/home/xingjia2@andrew.cmu.edu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/xingjia2@andrew.cmu.edu/.conda/envs/research-env/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/xingjia2@andrew.cmu.edu/.local/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Loading checkpoint shards: 100%|██████████| 2/2 [00:15<00:00,  8.00s/it]
/home/xingjia2@andrew.cmu.edu/.local/lib/python3.10/

Trainable params: 29,360,640
Trainable %: 0.4038%
Step 0 | Loss: 5.5004
Step 50 | Loss: 2.3772
Step 100 | Loss: 4.5257
Step 150 | Loss: 4.4995
Step 200 | Loss: 2.5779
Step 250 | Loss: 2.5299
Step 300 | Loss: 2.2384
Step 350 | Loss: 3.0112
Step 400 | Loss: 2.9213
Step 450 | Loss: 2.2783
✅ Training finished | Skipped batches: 297

=== ROUTING DIAGNOSTICS ===
Expert usage probs: [0.21529297530651093, 0.08359374850988388, 0.0768750011920929, 0.12306640297174454, 0.053105469793081284, 0.06230468675494194, 0.1489843726158142, 0.23677735030651093]
Expert-usage entropy: 1.9467
Max possible entropy: 2.0794
Top-2 routing entropy (max log2≈0.693): 0.6914


In [1]:
# ============================================================
# HYDRALORA — LONG TRAINING + ENTROPY LOGGING
# Mistral-7B | A100 80GB | BF16 | Paper-faithful
# ============================================================

import os, math
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:64"

import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from torch.utils.data import DataLoader

# -------------------------
# Setup
# -------------------------
device = "cuda"
torch.backends.cuda.matmul.allow_tf32 = True

model_name = "mistralai/Mistral-7B-v0.1"

# -------------------------
# Tokenizer
# -------------------------
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=False
)
tokenizer.pad_token = tokenizer.eos_token

# -------------------------
# Model
# -------------------------
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

model.config.pad_token_id = tokenizer.eos_token_id
model.config.use_flash_attention_2 = True
model.config.use_cache = False

model.gradient_checkpointing_enable()
model.enable_input_require_grads()

# ============================================================
# HydraLoRA (Top-2 Routing)
# ============================================================
class HydraLoRA(nn.Module):
    def __init__(
        self,
        in_features,
        out_features,
        r=8,
        num_experts=8,
        alpha=32,
        dropout=0.05,
        top_k=2
    ):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.scaling = alpha / r

        self.lora_A = nn.ModuleList([
            nn.Linear(in_features, r, bias=False)
            for _ in range(num_experts)
        ])
        self.lora_B = nn.ModuleList([
            nn.Linear(r, out_features, bias=False)
            for _ in range(num_experts)
        ])

        self.router = nn.Linear(in_features, num_experts)
        self.dropout = nn.Dropout(dropout)

        for A, B in zip(self.lora_A, self.lora_B):
            nn.init.kaiming_uniform_(A.weight, a=5 ** 0.5)
            nn.init.zeros_(B.weight)

        nn.init.zeros_(self.router.weight)
        nn.init.zeros_(self.router.bias)

    def forward(self, x):
        logits = self.router(x)
        logits = logits - logits.max(dim=-1, keepdim=True).values
        logits = torch.clamp(logits, -20, 20)

        topk_vals, topk_idx = torch.topk(logits, k=self.top_k, dim=-1)
        gates = F.softmax(topk_vals, dim=-1)

        expert_outs = []
        for A, B in zip(self.lora_A, self.lora_B):
            expert_outs.append(
                B(self.dropout(A(x))) * self.scaling
            )
        expert_outs = torch.stack(expert_outs, dim=-2)

        out = torch.zeros_like(expert_outs[..., 0, :])
        for i in range(self.top_k):
            idx = topk_idx[..., i]
            gate = gates[..., i].unsqueeze(-1)
            selected = torch.gather(
                expert_outs,
                dim=-2,
                index=idx.unsqueeze(-1).unsqueeze(-1)
                    .expand(-1, -1, 1, expert_outs.size(-1))
            ).squeeze(-2)
            out += gate * selected

        return out

class HydraLoRALinear(nn.Module):
    def __init__(self, base_layer):
        super().__init__()
        self.base = base_layer
        self.hydra = HydraLoRA(
            base_layer.in_features,
            base_layer.out_features
        ).to(
            device=base_layer.weight.device,
            dtype=base_layer.weight.dtype
        )

        for p in self.base.parameters():
            p.requires_grad = False

    def forward(self, x):
        return self.base(x) + self.hydra(x)

# -------------------------
# Inject into Q,V
# -------------------------
def apply_hydralora(model):
    for name, module in model.named_modules():
        if name.endswith("q_proj") or name.endswith("v_proj"):
            parent = model
            *path, attr = name.split(".")
            for p in path:
                parent = getattr(parent, p)
            setattr(parent, attr, HydraLoRALinear(getattr(parent, attr)))

apply_hydralora(model)

# -------------------------
# Freeze non-Hydra params
# -------------------------
for n, p in model.named_parameters():
    p.requires_grad = ("hydra" in n.lower())

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,}")
print(f"Trainable %: {100 * trainable / total:.4f}%")

# ============================================================
# Dataset
# ============================================================
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")

def tokenize_fn(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128,
        padding=False
    )

tokenized = dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=["text"]
)
tokenized.set_format(type="torch")

def collate_fn(batch):
    return {
        "input_ids": torch.nn.utils.rnn.pad_sequence(
            [x["input_ids"] for x in batch],
            batch_first=True,
            padding_value=tokenizer.pad_token_id
        )
    }

train_loader = DataLoader(
    tokenized["train"],
    batch_size=1,
    shuffle=True,
    collate_fn=collate_fn
)

# -------------------------
# Optimizer
# -------------------------
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=5e-5
)

# ============================================================
# Training (LONG)
# ============================================================
model.train()

GRAD_ACC = 8
MAX_STEPS = 3000
LOG_EVERY = 500
SAVE_EVERY = 1000

num_experts = 8
step = 0
skipped = 0

for batch in train_loader:
    input_ids = batch["input_ids"].to(device)

    if input_ids.numel() == 0 or input_ids.shape[1] < 2:
        skipped += 1
        continue

    outputs = model(input_ids=input_ids, labels=input_ids)
    loss = outputs.loss

    if torch.isnan(loss) or torch.isinf(loss):
        optimizer.zero_grad(set_to_none=True)
        skipped += 1
        continue

    loss = loss / GRAD_ACC
    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        filter(lambda p: p.requires_grad, model.parameters()),
        max_norm=1.0
    )

    if (step + 1) % GRAD_ACC == 0:
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)

    # -------------------------
    # Logging
    # -------------------------
    if step % LOG_EVERY == 0:
        print(f"Step {step} | Loss: {loss.item() * GRAD_ACC:.4f}")

        # ---- Expert-usage entropy ----
        expert_counts = torch.zeros(num_experts, device="cuda")

        with torch.no_grad():
            x = torch.randn(
                4, 32, model.config.hidden_size,
                device="cuda", dtype=torch.bfloat16
            )
            logits = model.model.layers[0].self_attn.q_proj.hydra.router(x)
            _, topk_idx = torch.topk(logits, k=2, dim=-1)

            for i in range(2):
                expert_counts.scatter_add_(
                    0,
                    topk_idx[..., i].reshape(-1),
                    torch.ones_like(
                        topk_idx[..., i].reshape(-1),
                        dtype=torch.float
                    )
                )

        probs = expert_counts / expert_counts.sum()
        entropy = -(probs * torch.log(probs + 1e-8)).sum()

        print(f"  Expert-usage entropy: {entropy.item():.4f}")

    # -------------------------
    # Checkpoint
    # -------------------------
    if step > 0 and step % SAVE_EVERY == 0:
        torch.save(
            model.state_dict(),
            f"hydralora_step_{step}.pt"
        )
        print(f"💾 Saved checkpoint at step {step}")

    step += 1
    if step >= MAX_STEPS:
        break

print(f"\n✅ Training finished | Skipped batches: {skipped}")


/home/xingjia2@andrew.cmu.edu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/xingjia2@andrew.cmu.edu/.conda/envs/research-env/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/xingjia2@andrew.cmu.edu/.local/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Loading checkpoint shards: 100%|██████████| 2/2 [00:04<00:00,  2.39s/it]
/home/xingjia2@andrew.cmu.edu/.local/lib/python3.10/

Trainable params: 29,360,640
Trainable %: 0.4038%
Step 0 | Loss: 5.6741
  Expert-usage entropy: 0.6931
Step 500 | Loss: 2.6134
  Expert-usage entropy: 1.8873
Step 1000 | Loss: 2.0346
  Expert-usage entropy: 1.8978
💾 Saved checkpoint at step 1000
Step 1500 | Loss: 2.0252
  Expert-usage entropy: 1.9079
Step 2000 | Loss: 1.5657
  Expert-usage entropy: 1.8897
💾 Saved checkpoint at step 2000
Step 2500 | Loss: 1.6705
  Expert-usage entropy: 1.9544

✅ Training finished | Skipped batches: 1723


In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    get_linear_schedule_with_warmup
)
from datasets import load_dataset

# =========================
# CONFIG
# =========================
MODEL_NAME = "mistralai/Mistral-7B-v0.1"
MAX_LENGTH = 512
BATCH_SIZE = 1
GRAD_ACC = 8
MAX_STEPS = 1500
LR = 2e-4
NUM_EXPERTS = 8
LORA_RANK = 8
DEVICE = "cuda"

torch.backends.cuda.matmul.allow_tf32 = True

# =========================
# TOKENIZER (FAST OFF)
# =========================
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=False
)
tokenizer.pad_token = tokenizer.eos_token

# =========================
# DATASET (CORRECT SPLIT)
# =========================
dataset = load_dataset(
    "HuggingFaceH4/ultrachat_200k",
    split="train_sft[:50000]"
)

def preprocess(example):
    user, assistant = "", ""
    for m in example["messages"]:
        if m["role"] == "user":
            user += m["content"]
        elif m["role"] == "assistant":
            assistant += m["content"]

    text = f"<s>[INST] {user.strip()} [/INST] {assistant.strip()}</s>"

    tokens = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False
    )

    input_ids = tokens["input_ids"]
    labels = input_ids.copy()

    inst_tokens = tokenizer(
        "[/INST]",
        add_special_tokens=False
    )["input_ids"]

    inst_end = None
    for i in range(len(input_ids) - len(inst_tokens)):
        if input_ids[i:i + len(inst_tokens)] == inst_tokens:
            inst_end = i + len(inst_tokens)
            break

    if inst_end is not None:
        labels[:inst_end] = [-100] * inst_end

    return {
        "input_ids": input_ids,
        "labels": labels
    }

dataset = dataset.map(
    preprocess,
    remove_columns=dataset.column_names
)

def collate_fn(batch):
    return {
        "input_ids": torch.tensor(
            [x["input_ids"] for x in batch],
            dtype=torch.long
        ),
        "labels": torch.tensor(
            [x["labels"] for x in batch],
            dtype=torch.long
        )
    }

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

# =========================
# HYDRALORA MODULES
# =========================
class HydraLoRA(nn.Module):
    def __init__(self, in_dim, out_dim, r, num_experts):
        super().__init__()
        self.router = nn.Linear(in_dim, num_experts, bias=False)

        self.lora_A = nn.ModuleList([
            nn.Linear(in_dim, r, bias=False) for _ in range(num_experts)
        ])
        self.lora_B = nn.ModuleList([
            nn.Linear(r, out_dim, bias=False) for _ in range(num_experts)
        ])

        for A, B in zip(self.lora_A, self.lora_B):
            nn.init.kaiming_uniform_(A.weight, a=5 ** 0.5)
            nn.init.zeros_(B.weight)

    def forward(self, x):
        gates = F.softmax(self.router(x), dim=-1)
        out = 0
        for i, (A, B) in enumerate(zip(self.lora_A, self.lora_B)):
            out = out + gates[..., i:i+1] * B(A(x))
        return out

class HydraLoRALinear(nn.Module):
    def __init__(self, base_layer, r, num_experts):
        super().__init__()
        self.base = base_layer
        self.hydra = HydraLoRA(
            base_layer.in_features,
            base_layer.out_features,
            r,
            num_experts
        )

    def forward(self, x):
        return self.base(x) + self.hydra(x)

# =========================
# LOAD MODEL
# =========================
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map={"": DEVICE}
)

# Freeze backbone
for p in model.parameters():
    p.requires_grad = False

# Inject HydraLoRA
for layer in model.model.layers:
    layer.self_attn.q_proj = HydraLoRALinear(
        layer.self_attn.q_proj,
        LORA_RANK,
        NUM_EXPERTS
    ).to(DEVICE, dtype=torch.bfloat16)

    layer.self_attn.v_proj = HydraLoRALinear(
        layer.self_attn.v_proj,
        LORA_RANK,
        NUM_EXPERTS
    ).to(DEVICE, dtype=torch.bfloat16)

# =========================
# PARAM COUNT
# =========================
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,}")
print(f"Trainable %: {100 * trainable / total:.4f}%")

# =========================
# OPTIMIZER
# =========================
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR
)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=100,
    num_training_steps=MAX_STEPS
)

# =========================
# TRAIN LOOP (STOP @ 1500)
# =========================
model.train()
optimizer.zero_grad()
step = 0

for batch in loader:
    batch = {k: v.to(DEVICE) for k, v in batch.items()}

    outputs = model(**batch)
    loss = outputs.loss / GRAD_ACC
    loss.backward()

    if (step + 1) % GRAD_ACC == 0:
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

    if step % 500 == 0:
        print(f"Step {step} | Loss: {loss.item() * GRAD_ACC:.4f}")

    step += 1
    if step >= MAX_STEPS:
        break

# =========================
# SAVE BEST CHECKPOINT
# =========================
torch.save(model.state_dict(), "hydralora_step1500.pt")
print("✅ Training stopped at step 1500 — checkpoint saved")


Loading checkpoint shards: 100%|██████████| 2/2 [00:05<00:00,  2.59s/it]


Trainable params: 29,360,128
Trainable %: 0.4038%
Step 0 | Loss: 1.9110
Step 500 | Loss: 0.7705
Step 1000 | Loss: 0.9225
✅ Training stopped at step 1500 — checkpoint saved


In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
import numpy as np
from tqdm import tqdm
from typing import Dict, List, Optional, Tuple
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ==================================================
# 优化后的 HydraLoRA 实现
# ==================================================

class OptimizedHydraLoraLinear(nn.Module):
    def __init__(self, base_layer, num_experts=8, lora_rank=8, expert_dropout=0.1, 
                 router_temperature=1.0, load_balancing_lambda=0.01):
        super().__init__()
        self.base_layer = base_layer
        
        # 获取输入输出特征
        self.in_features = base_layer.in_features
        self.out_features = base_layer.out_features
        self.num_experts = num_experts
        self.lora_rank = lora_rank
        self.expert_dropout = expert_dropout
        self.router_temperature = router_temperature
        self.load_balancing_lambda = load_balancing_lambda
        
        # 保存原始权重
        self.weight = base_layer.weight
        self.bias = base_layer.bias if hasattr(base_layer, 'bias') else None
        
        # Router 参数
        self.router = nn.Linear(self.in_features, num_experts, bias=False)
        nn.init.normal_(self.router.weight, mean=0.0, std=0.02)
        
        # LoRA 参数
        self.lora_A = nn.ParameterList([
            nn.Parameter(torch.randn(lora_rank, self.in_features) * 0.02)
            for _ in range(num_experts)
        ])
        self.lora_B = nn.ParameterList([
            nn.Parameter(torch.zeros(self.out_features, lora_rank))
            for _ in range(num_experts)
        ])
        
        # 用于负载均衡的辅助变量
        self.register_buffer('expert_usage', torch.zeros(num_experts))
        self.register_buffer('total_samples', torch.tensor(0.0))
        
        # 禁用基础层梯度
        for param in base_layer.parameters():
            param.requires_grad = False
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        优化后的前向传播，包含负载均衡
        """
        # 基础层输出
        base_output = F.linear(x, self.weight, self.bias)
        
        # 保存原始形状
        original_shape = x.shape
        if len(original_shape) == 3:
            batch_size, seq_len, _ = original_shape
            x_flat = x.reshape(-1, self.in_features)
        else:
            x_flat = x
            batch_size = x.shape[0]
        
        # Router 计算（带温度参数）
        router_logits = self.router(x_flat) / self.router_temperature
        router_weights = F.softmax(router_logits, dim=-1)
        
        # 计算专家选择（用于负载均衡）
        expert_indices = torch.argmax(router_logits, dim=-1)
        
        # 更新专家使用统计（训练时）
        if self.training:
            # 计算每个专家的使用频率
            expert_counts = torch.bincount(
                expert_indices.flatten(), 
                minlength=self.num_experts
            ).float()
            
            # 指数移动平均更新
            decay = 0.9
            self.expert_usage = decay * self.expert_usage + (1 - decay) * expert_counts
            self.total_samples = decay * self.total_samples + (1 - decay) * x_flat.shape[0]
        
        # 计算 LoRA 输出
        lora_output = torch.zeros_like(base_output)
        
        # 批处理计算所有专家的贡献
        for i in range(self.num_experts):
            # 获取当前专家的权重
            expert_weight = router_weights[:, i].unsqueeze(-1)  # [batch_seq, 1]
            
            # 计算当前专家的 LoRA 输出
            expert_output = F.linear(
                F.linear(x_flat, self.lora_A[i].t()), 
                self.lora_B[i].t()
            )
            
            # 加权求和
            lora_output = lora_output + expert_weight * expert_output
        
        # 负载均衡损失（训练时）
        load_balancing_loss = 0.0
        if self.training and self.load_balancing_lambda > 0:
            # 计算专家使用方差
            expert_usage_prob = self.expert_usage / (self.total_samples + 1e-8)
            load_balancing_loss = torch.var(expert_usage_prob) * self.load_balancing_lambda
            self.load_balancing_loss = load_balancing_loss
        
        # 重塑输出形状
        if len(original_shape) == 3:
            lora_output = lora_output.reshape(batch_size, seq_len, self.out_features)
        
        return base_output + lora_output
    
    def get_expert_usage_stats(self):
        """获取专家使用统计"""
        if self.total_samples > 0:
            usage_prob = self.expert_usage / self.total_samples
            entropy = -torch.sum(usage_prob * torch.log(usage_prob + 1e-8))
            max_entropy = torch.log(torch.tensor(float(self.num_experts)))
            return {
                'usage_prob': usage_prob.cpu().numpy(),
                'entropy': entropy.item(),
                'max_entropy': max_entropy.item(),
                'normalized_entropy': entropy.item() / max_entropy.item()
            }
        return None

# ==================================================
# 高级 HydraLoRA 包装器
# ==================================================

class AdvancedHydraLoRA:
    def __init__(self, model_path: str, checkpoint_path: str = None):
        print(f"Loading model from {model_path}")
        
        # 加载基础模型
        self.model = AutoModelForCausalLM.from_pretrained(
            model_path,
            torch_dtype=torch.float16,
            device_map="auto"
        )
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        
        # 存储 HydraLoRA 层
        self.hydralora_layers = []
        
        # 加载 checkpoint（如果提供）
        if checkpoint_path:
            self.load_checkpoint(checkpoint_path)
    
    def convert_to_hydralora(self, num_experts=8, lora_rank=8, 
                             target_modules=['q_proj', 'v_proj']):
        """将模型转换为 HydraLoRA"""
        print(f"\nConverting to HydraLoRA with {num_experts} experts...")
        
        converted_layers = []
        for name, module in self.model.named_modules():
            # 检查是否为目标模块
            is_target = any(target in name for target in target_modules)
            is_linear = isinstance(module, nn.Linear)
            is_attention = 'attention' in name.lower()
            
            if is_target and is_linear and is_attention:
                # 获取父模块
                parent_name = '.'.join(name.split('.')[:-1])
                child_name = name.split('.')[-1]
                parent = self._get_module(parent_name)
                
                if parent is not None:
                    # 创建 HydraLoRA 层
                    hydralora_layer = OptimizedHydraLoraLinear(
                        base_layer=module,
                        num_experts=num_experts,
                        lora_rank=lora_rank,
                        expert_dropout=0.1,
                        router_temperature=1.0,
                        load_balancing_lambda=0.01
                    )
                    
                    # 替换层
                    setattr(parent, child_name, hydralora_layer)
                    converted_layers.append((name, hydralora_layer))
                    
                    print(f"  ✓ Converted {name}")
        
        self.hydralora_layers = converted_layers
        print(f"\nTotal converted layers: {len(converted_layers)}")
        return converted_layers
    
    def _get_module(self, path: str):
        """通过路径获取模块"""
        module = self.model
        for part in path.split('.'):
            if part:
                module = getattr(module, part)
        return module
    
    def load_checkpoint(self, checkpoint_path: str):
        """加载 checkpoint"""
        print(f"\nLoading checkpoint from {checkpoint_path}")
        
        try:
            checkpoint = torch.load(checkpoint_path, map_location='cpu')
            
            # 直接加载到模型
            load_result = self.model.load_state_dict(checkpoint, strict=False)
            
            print(f"✅ Checkpoint loaded")
            print(f"   Missing keys: {len(load_result.missing_keys)}")
            print(f"   Unexpected keys: {len(load_result.unexpected_keys)}")
            
            if load_result.missing_keys:
                print(f"\nMissing keys (first 5):")
                for key in list(load_result.missing_keys)[:5]:
                    print(f"  - {key}")
            
            return True
            
        except Exception as e:
            print(f"❌ Failed to load checkpoint: {e}")
            return False
    
    def analyze_routing(self, num_samples=100, batch_size=4):
        """分析路由行为"""
        print("\n" + "="*60)
        print("ANALYZING ROUTING BEHAVIOR")
        print("="*60)
        
        # 收集统计数据
        all_expert_probs = []
        all_entropies = []
        
        self.model.eval()
        
        with torch.no_grad():
            for i in tqdm(range(0, num_samples, batch_size), desc="Analyzing"):
                # 生成随机输入
                input_ids = torch.randint(0, 1000, (batch_size, 32)).to(self.model.device)
                attention_mask = torch.ones_like(input_ids)
                
                # 前向传播
                outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
                
                # 收集每层的路由统计
                for name, layer in self.hydralora_layers:
                    stats = layer.get_expert_usage_stats()
                    if stats:
                        all_expert_probs.append(stats['usage_prob'])
                        all_entropies.append(stats['entropy'])
        
        if not all_expert_probs:
            print("No HydraLoRA layers found to analyze")
            return
        
        # 计算平均统计
        avg_expert_probs = np.mean(all_expert_probs, axis=0)
        avg_entropy = np.mean(all_entropies)
        max_entropy = np.log(len(avg_expert_probs))
        
        print(f"\n=== ROUTING DIAGNOSTICS ===")
        print(f"\nExpert usage probabilities:")
        for i, prob in enumerate(avg_expert_probs):
            print(f"  Expert {i}: {prob:.4f}")
        
        print(f"\nExpert-usage entropy: {avg_entropy:.4f}")
        print(f"Max possible entropy: {max_entropy:.4f}")
        print(f"Normalized entropy: {avg_entropy/max_entropy:.4f}")
        print(f"Expected (paper-like): ~1.2 – 1.6")
        
        # 可视化
        self._plot_routing_stats(avg_expert_probs, avg_entropy, max_entropy)
        
        # 提供分析建议
        self._provide_routing_advice(avg_expert_probs, avg_entropy, max_entropy)
    
    def _plot_routing_stats(self, expert_probs, entropy, max_entropy):
        """绘制路由统计图"""
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        
        # 专家使用概率
        axes[0].bar(range(len(expert_probs)), expert_probs)
        axes[0].set_xlabel('Expert Index')
        axes[0].set_ylabel('Usage Probability')
        axes[0].set_title('Expert Usage Distribution')
        axes[0].grid(True, alpha=0.3)
        
        # 熵值比较
        categories = ['Current', 'Max Possible', 'Paper Target']
        values = [entropy, max_entropy, 1.4]  # 1.4 是论文目标值
        axes[1].bar(categories, values, color=['blue', 'gray', 'green'])
        axes[1].set_ylabel('Entropy')
        axes[1].set_title('Routing Entropy Comparison')
        axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig('routing_analysis.png', dpi=150, bbox_inches='tight')
        print(f"\n📊 Routing analysis plot saved to 'routing_analysis.png'")
    
    def _provide_routing_advice(self, expert_probs, entropy, max_entropy):
        """提供路由优化建议"""
        print(f"\n=== ROUTING OPTIMIZATION ADVICE ===")
        
        normalized_entropy = entropy / max_entropy
        
        # 分析专家使用分布
        max_prob = np.max(expert_probs)
        min_prob = np.min(expert_probs)
        std_prob = np.std(expert_probs)
        
        print(f"\n1. Distribution Analysis:")
        print(f"   - Most used expert: {max_prob:.4f}")
        print(f"   - Least used expert: {min_prob:.4f}")
        print(f"   - Standard deviation: {std_prob:.4f}")
        
        # 提供建议
        if normalized_entropy > 0.95:
            print(f"\n2. ⚠️  Routing is too uniform:")
            print(f"   - Consider increasing router_temperature")
            print(f"   - Try higher load_balancing_lambda")
            print(f"   - Reduce number of experts if needed")
        elif normalized_entropy < 0.6:
            print(f"\n2. ⚠️  Routing is too concentrated:")
            print(f"   - Consider decreasing router_temperature")
            print(f"   - Try lower load_balancing_lambda")
            print(f"   - Experts may need better initialization")
        else:
            print(f"\n2. ✅ Routing looks good!")
            print(f"   - Entropy is in reasonable range")
        
        # 检查负载均衡
        if std_prob > 0.15:
            print(f"\n3. ⚠️  Load imbalance detected:")
            print(f"   - Some experts are underutilized")
            print(f"   - Consider increasing load_balancing_lambda")
        else:
            print(f"\n3. ✅ Load balance is good")
    
    def tune_routing_parameters(self, initial_lr=1e-3, num_steps=100):
        """微调路由参数"""
        print(f"\n=== TUNING ROUTING PARAMETERS ===")
        
        # 只训练 router 参数
        trainable_params = []
        for _, layer in self.hydralora_layers:
            trainable_params.append({'params': layer.router.parameters()})
        
        if not trainable_params:
            print("No trainable parameters found")
            return
        
        optimizer = torch.optim.Adam(trainable_params, lr=initial_lr)
        
        print(f"Training {len(trainable_params)} router parameters...")
        
        losses = []
        for step in tqdm(range(num_steps), desc="Tuning"):
            # 生成训练数据
            inputs = torch.randn(32, 128, self.hydralora_layers[0][1].in_features).to(self.model.device)
            
            # 计算负载均衡损失
            total_loss = 0
            for _, layer in self.hydralora_layers:
                layer.train()
                _ = layer(inputs)  # 前向传播以计算损失
                if hasattr(layer, 'load_balancing_loss'):
                    total_loss += layer.load_balancing_loss
            
            # 反向传播
            optimizer.zero_grad()
            total_loss.backward()
            optimizer.step()
            
            losses.append(total_loss.item())
        
        print(f"Final loss: {losses[-1]:.6f}")
        return losses
    
    def evaluate_performance(self, test_dataset=None, num_samples=20):
        """评估模型性能"""
        print(f"\n=== PERFORMANCE EVALUATION ===")
        
        if test_dataset is None:
            # 使用默认测试提示
            test_prompts = [
                "Explain the concept of machine learning in simple terms.",
                "What are the benefits of renewable energy sources?",
                "Write a short story about a robot who learns to paint.",
                "How does the human brain process visual information?",
                "Describe the water cycle in detail."
            ]
        else:
            test_prompts = test_dataset[:num_samples]
        
        print(f"Testing on {len(test_prompts)} prompts...")
        
        results = []
        for i, prompt in enumerate(test_prompts):
            try:
                # 编码输入
                inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256)
                inputs = {k: v.to(self.model.device) for k, v in inputs.items()}
                
                # 生成输出
                with torch.no_grad():
                    outputs = self.model.generate(
                        **inputs,
                        max_length=min(512, len(prompt.split()) + 100),
                        num_return_sequences=1,
                        temperature=0.7,
                        do_sample=True,
                        pad_token_id=self.tokenizer.pad_token_id
                    )
                
                generated = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
                
                results.append({
                    'prompt': prompt,
                    'generated': generated,
                    'length': len(generated.split())
                })
                
                print(f"\nPrompt {i+1}: {prompt[:50]}...")
                print(f"Generated: {generated[:100]}...")
                
            except Exception as e:
                print(f"Error on prompt {i+1}: {e}")
        
        return results
    
    def save_model(self, save_path: str):
        """保存模型"""
        print(f"\nSaving model to {save_path}")
        self.model.save_pretrained(save_path)
        self.tokenizer.save_pretrained(save_path)
        print(f"✅ Model saved successfully")

# ==================================================
# 主函数
# ==================================================

def main():
    # 配置
    MODEL_PATH = "NousResearch/Meta-Llama-3.1-8B"
    CHECKPOINT_PATH = "hydralora_step1500.pt"
    
    print("="*60)
    print("HYDRALORA ADVANCED ANALYSIS")
    print("="*60)
    
    # 初始化
    hydralora = AdvancedHydraLoRA(
        model_path=MODEL_PATH,
        checkpoint_path=CHECKPOINT_PATH
    )
    
    # 转换为 HydraLoRA（如果 checkpoint 已包含则跳过）
    # hydralora.convert_to_hydralora(num_experts=8, lora_rank=8)
    
    # 分析路由行为
    print("\nAnalyzing routing behavior...")
    hydralora.analyze_routing(num_samples=50, batch_size=2)
    
    # 微调路由参数（可选）
    tune = input("\nDo you want to tune routing parameters? (y/n): ")
    if tune.lower() == 'y':
        losses = hydralora.tune_routing_parameters(num_steps=50)
        print("Re-analyzing after tuning...")
        hydralora.analyze_routing(num_samples=30, batch_size=2)
    
    # 评估性能
    print("\nEvaluating model performance...")
    results = hydralora.evaluate_performance(num_samples=5)
    
    # 保存模型（可选）
    save = input("\nDo you want to save the model? (y/n): ")
    if save.lower() == 'y':
        save_path = "hydralora_tuned_model"
        hydralora.save_model(save_path)
    
    print("\n" + "="*60)
    print("ANALYSIS COMPLETE")
    print("="*60)
    
    # 总结
    print("\n📋 Summary:")
    print(f"- Model: {MODEL_PATH}")
    print(f"- Checkpoint: {CHECKPOINT_PATH}")
    print(f"- HydraLoRA layers: {len(hydralora.hydralora_layers)}")
    print(f"- Analysis plots saved to 'routing_analysis.png'")

if __name__ == "__main__":
    main()

HYDRALORA ADVANCED ANALYSIS
Loading model from NousResearch/Meta-Llama-3.1-8B


ValueError: `rope_scaling` must be a dictionary with with two fields, `type` and `factor`, got {'factor': 8.0, 'low_freq_factor': 1.0, 'high_freq_factor': 4.0, 'original_max_position_embeddings': 8192, 'rope_type': 'llama3'}